# Red Scare results
Running our algorithms for `RedScare2026`. Each algorithm saves one CSV in `results/` and prints
runtime totals and averages by graph family. Timing excludes graph loading.
The last cell combines the saved files.


In [1]:
from pathlib import Path
from importlib import reload
from time import perf_counter
import pandas as pd

#we read in each file of the data by
files = sorted(Path('data').glob('*.txt'))

#for creating the results-files, we need to first create a folder:
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)

#And then determine the format of the results-files:
problem_files = {'alternate': '01_alternate.csv',
                 'few': '02_few.csv',
                 'many': '03_many.csv',
                 'none': '04_none.csv',
                 'some': '05_some.csv'}
file_columns = ['file_name', 'graph_type', 'n_nodes', 'result', 'runtime_seconds']

#to identify the types of graphs in the data files, we need:
graph_types = {'bht': 'Undirected',
               'common': 'Undirected word-graph',
               'dodecahedron': 'Undirected all red',
               'G': 'Undirected small example',
               'gnm': 'Undirected random',
               'grid': 'Undirected grid',
               'increase': 'Directed Acyclic Graph',
               'P3': '3-node path',
               'rusty': 'Undirected word-graph',
               'ski': 'Directed Acyclic Graph',
               'smallworld': 'Undirected with high clustering',
               'wall-n': 'Undirected 8-node blocks separated',
               'wall-p': 'Undirected 8-node blocks overlapping',
               'wall-z': 'Undirected 8-node blocks touching corners'}

#Now, to map from the file names to the graph types, we need to extract the 
# relevant part of the file name. The following dictionary comprehension does that:
#THIS ONLY MAKES SENSE BECAUSE WE LATER ITERATE OVER THE FILES FOR EACH ALGORITHM,
# OTHERWISE IT WOULD BE BETTER TO DO IT INSIDE THE ALGORITHM FUNCTION
file_types = {file: '-'.join(file.stem.split('-')[:2]) if file.stem.startswith('wall-')
              else file.stem.split('-')[0]
              for file in files}

def run_algorithm(name, reader, solver):
    #initialise the rows-list which will contain the results for each file
    rows = []

    #iterate the requested algorithm over all the files in the data-folder
    for file in files:

        #Get the graph-dict, start and end nodes, and the red edges from the file using the reader function
        graph, s, t, red, n = reader(file)

        #start the algorithm-timer, run the algorithm, and stop the timer
        start = perf_counter()
        result = solver(graph, s, t, red)
        elapsed = perf_counter() - start

        #fill up a line in rows with the file-name, graph-type based on extracted file-type, number of nodes, result of the algorithm, and the elapsed time
        rows.append([file.name, graph_types[file_types[file]], n, result, elapsed])

    #save the results to csv-files
    df_rows = pd.DataFrame(rows, columns=file_columns)
    df_rows.to_csv(results_dir / problem_files[name], index=False)

    #print out the average and total times for each file-type
    timings = df_rows.groupby([file_types[file] for file in files])['runtime_seconds'].agg(['count', 'sum', 'mean'])

    #print out the timings for each graph-type, with file-type left-aligned, count right-aligned, total and average times formatted to 4 decimal places
    for file_type, (count, total, avg) in timings.iterrows():
        print(f'{file_type:<14} {count:>3.0f} graphs , total: {total:.4f} s , avg {avg:.4f} s')


# Alternate

In [2]:
from AlternateProblem import read_graph_alternating, bfs_alternating

run_algorithm('alternate', read_graph_alternating, bfs_alternating)


G                1 graphs , total: 0.0000 s , avg 0.0000 s
P3               1 graphs , total: 0.0000 s , avg 0.0000 s
bht              1 graphs , total: 0.0000 s , avg 0.0000 s
common          30 graphs , total: 0.0399 s , avg 0.0013 s
dodecahedron     1 graphs , total: 0.0000 s , avg 0.0000 s
gnm             24 graphs , total: 0.0039 s , avg 0.0002 s
grid            12 graphs , total: 0.0046 s , avg 0.0004 s
increase        18 graphs , total: 0.0003 s , avg 0.0000 s
rusty           17 graphs , total: 0.0002 s , avg 0.0000 s
ski             13 graphs , total: 0.0000 s , avg 0.0000 s
smallworld      12 graphs , total: 0.0014 s , avg 0.0001 s
wall-n           8 graphs , total: 0.0001 s , avg 0.0000 s
wall-p           8 graphs , total: 0.0001 s , avg 0.0000 s
wall-z           8 graphs , total: 0.0001 s , avg 0.0000 s


# Few

In [ ]:
from FewProblem import read_graph_few, dijkstra_few

run_algorithm('few', read_graph_few, dijkstra_few)

G                1 graphs , total: 0.0000 s , avg 0.0000 s
P3               1 graphs , total: 0.0000 s , avg 0.0000 s
bht              1 graphs , total: 0.0076 s , avg 0.0076 s
common          30 graphs , total: 0.2505 s , avg 0.0084 s
dodecahedron     1 graphs , total: 0.0000 s , avg 0.0000 s
gnm             24 graphs , total: 0.0328 s , avg 0.0014 s
grid            12 graphs , total: 0.0125 s , avg 0.0010 s
increase        18 graphs , total: 0.0126 s , avg 0.0007 s
rusty           17 graphs , total: 0.0889 s , avg 0.0052 s
ski             13 graphs , total: 0.0003 s , avg 0.0000 s
smallworld      12 graphs , total: 0.0130 s , avg 0.0011 s
wall-n           8 graphs , total: 0.0001 s , avg 0.0000 s
wall-p           8 graphs , total: 0.0001 s , avg 0.0000 s
wall-z           8 graphs , total: 0.0001 s , avg 0.0000 s


# Many

In [ ]:
from ManyProblem import read_graph_many

df_many = pd.DataFrame([[file.name, graph_types[file_types[file]], read_graph_many(file)[4], '?!', 'no time'] for file in files], columns=file_columns)
df_many.to_csv(results_dir / problem_files['many'], index=False)
print('Many: no solver implemented yet.')

Many: no solver implemented yet.


# None

In [ ]:
from NoneProblem import read_graph_none, bfs_none

run_algorithm('none', read_graph_none, bfs_none) 

G                1 graphs , total: 0.0000 s , avg 0.0000 s
P3               1 graphs , total: 0.0000 s , avg 0.0000 s
bht              1 graphs , total: 0.0018 s , avg 0.0018 s
common          30 graphs , total: 0.0305 s , avg 0.0010 s
dodecahedron     1 graphs , total: 0.0000 s , avg 0.0000 s
gnm             24 graphs , total: 0.0063 s , avg 0.0003 s
grid            12 graphs , total: 0.0027 s , avg 0.0002 s
increase        18 graphs , total: 0.0003 s , avg 0.0000 s
rusty           17 graphs , total: 0.0589 s , avg 0.0035 s
ski             13 graphs , total: 0.0001 s , avg 0.0000 s
smallworld      12 graphs , total: 0.0036 s , avg 0.0003 s
wall-n           8 graphs , total: 0.0001 s , avg 0.0000 s
wall-p           8 graphs , total: 0.0001 s , avg 0.0000 s
wall-z           8 graphs , total: 0.0001 s , avg 0.0000 s


# Some

In [ ]:
from SomeProblem import read_graph_some, some_function

run_algorithm('some', read_graph_some, some_function)

G                1 graphs , total: 0.0000 s , avg 0.0000 s
P3               1 graphs , total: 0.0000 s , avg 0.0000 s
bht              1 graphs , total: 0.0004 s , avg 0.0004 s
common          30 graphs , total: 0.2647 s , avg 0.0088 s
dodecahedron     1 graphs , total: 0.0001 s , avg 0.0001 s
gnm             24 graphs , total: 2.3645 s , avg 0.0985 s
grid            12 graphs , total: 0.0159 s , avg 0.0013 s
increase        18 graphs , total: 0.0059 s , avg 0.0003 s
rusty           17 graphs , total: 0.0610 s , avg 0.0036 s
ski             13 graphs , total: 0.0005 s , avg 0.0000 s
smallworld      12 graphs , total: 0.0086 s , avg 0.0007 s
wall-n           8 graphs , total: 0.2469 s , avg 0.0309 s
wall-p           8 graphs , total: 0.1832 s , avg 0.0229 s
wall-z           8 graphs , total: 0.2103 s , avg 0.0263 s


## Combine saved results into results.txt

In [ ]:
# Collect each algorithm's result and each file's node count from the saved CSVs
df_alt = pd.read_csv('results/01_alternate.csv') #from which we need file_name, n_nodes, result
df_few = pd.read_csv('results/02_few.csv') #from which we need result
df_many = pd.read_csv('results/03_many.csv') #from which we need result
df_none = pd.read_csv('results/04_none.csv') #from which we need result
df_some = pd.read_csv('results/05_some.csv') #from which we need result

df_combined = df_alt[['file_name', 'n_nodes', 'result']].rename(columns={'result': 'A'}) \
    .merge(df_few[['file_name', 'result']].rename(columns={'result': 'F'}), on='file_name') \
        .merge(df_many[['file_name', 'result']].rename(columns={'result': 'M'}), on='file_name') \
            .merge(df_none[['file_name', 'result']].rename(columns={'result': 'N'}), on='file_name') \
                .merge(df_some[['file_name', 'result']].rename(columns={'result': 'S'}), on='file_name')

df_combined['file_name'] = df_combined['file_name'].str.removesuffix('.txt')
df_combined = df_combined.rename(columns={'file_name': 'Instance name', 'n_nodes': 'n'})

df_combined.to_csv('results.txt', sep='\t', index=False)